# BPT Diagram Sandbox

Can our CFM models reproduce the Kauffmann et al. (2003) BPT diagram?

## Pipeline
1. Same preprocessing as `FSFV2_compare_image_vs_photometry_experimental.ipynb`
2. Apply SNR > 3 cut on all four BPT emission lines (H$\alpha$, H$\beta$, [OIII] 5007, [NII] 6584)
3. Ground-truth BPT diagram from spectroscopic fluxes
4. Model-predicted BPT diagrams:
   - Image+Phot model (posterior medians)
   - Photometry-only model (posterior medians)
   - Image+Phot model (single trajectory)
   - Photometry-only model (single trajectory)

In [ ]:
from cfm_models import ImprovedConditionalFlowModel, ImprovedPhotometryConditionalFlowModelSimple
from cfm_utils import sample_properties_distribution_rk4, sample_properties_rk4

import numpy as np
import pandas as pd
from matplotlib import pyplot as plt
from pathlib import Path
from astropy.io import fits
import h5py
import os
import torch
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm

from sklearn.model_selection import train_test_split

# Preprocessing (identical to training notebooks)

In [ ]:
# Load the FastSpecFit FITS file
fits_path = Path("fastspec-iron.fits")

def to_native(arr):
    """Ensure a NumPy array is in native-endian dtype."""
    a = np.asarray(arr)
    if a.dtype.kind in "fiu":
        if a.dtype.byteorder == '>' or (a.dtype.byteorder == '=' and not np.little_endian):
            return a.byteswap().view(a.dtype.newbyteorder("="))
    return a

print("Loading FastSpecFit FITS file and applying preprocessing...")

In [ ]:
# Load and filter FITS data (including FLUX_IVAR for SNR cuts)
print("\nLoading HDU[1] (FASTSPEC) properties...")
with fits.open(fits_path, memmap=True) as hdul:
    fastspec_data = hdul[1].data
    
    print("Loading columns: Z, LOGMSTAR, SFR, DN4000, AV, emission line fluxes + IVARs")
    
    df_hdu1 = pd.DataFrame({
        'TARGETID': to_native(fastspec_data['TARGETID']),
        'Z': to_native(fastspec_data['Z']),
        'LOGMSTAR': to_native(fastspec_data['LOGMSTAR']),
        'SFR': to_native(fastspec_data['SFR']),
        'DN4000': to_native(fastspec_data['DN4000']),
        'AV': to_native(fastspec_data['AV']),
        'HBETA_FLUX': to_native(fastspec_data['HBETA_FLUX']),
        'OIII_5007_FLUX': to_native(fastspec_data['OIII_5007_FLUX']),
        'HALPHA_FLUX': to_native(fastspec_data['HALPHA_FLUX']),
        'NII_6584_FLUX': to_native(fastspec_data['NII_6584_FLUX']),
        # IVAR columns for SNR computation
        'HBETA_FLUX_IVAR': to_native(fastspec_data['HBETA_FLUX_IVAR']),
        'OIII_5007_FLUX_IVAR': to_native(fastspec_data['OIII_5007_FLUX_IVAR']),
        'HALPHA_FLUX_IVAR': to_native(fastspec_data['HALPHA_FLUX_IVAR']),
        'NII_6584_FLUX_IVAR': to_native(fastspec_data['NII_6584_FLUX_IVAR'])
    })
    
    print(f"Loaded {len(df_hdu1)} galaxies from HDU[1]")

print("\nLoading HDU[2] (METADATA) photometry...")
with fits.open(fits_path, memmap=True) as hdul:
    metadata_data = hdul[2].data
    
    df_hdu2 = pd.DataFrame({
        'TARGETID': to_native(metadata_data['TARGETID']),
        'SPECTYPE': np.char.strip(metadata_data['SPECTYPE'].astype(str)),
        'ZWARN': to_native(metadata_data['ZWARN']),
        'FLUX_G': to_native(metadata_data['FLUX_G']),
        'FLUX_R': to_native(metadata_data['FLUX_R']),
        'FLUX_Z': to_native(metadata_data['FLUX_Z']),
        'FLUX_W1': to_native(metadata_data['FLUX_W1']),
        'FLUX_W2': to_native(metadata_data['FLUX_W2'])
    })
    
    print(f"Loaded {len(df_hdu2)} galaxies from HDU[2]")

# Check for duplicate TARGETIDs
print("\nChecking for duplicate TARGETIDs...")
n_dup_hdu1 = df_hdu1['TARGETID'].duplicated().sum()
n_dup_hdu2 = df_hdu2['TARGETID'].duplicated().sum()
print(f"  HDU[1] duplicates: {n_dup_hdu1}")
print(f"  HDU[2] duplicates: {n_dup_hdu2}")

if n_dup_hdu1 > 0:
    df_hdu1 = df_hdu1.drop_duplicates(subset='TARGETID', keep='first').reset_index(drop=True)
if n_dup_hdu2 > 0:
    df_hdu2 = df_hdu2.drop_duplicates(subset='TARGETID', keep='first').reset_index(drop=True)

# Merge
print("\nMerging HDU[1] and HDU[2] by TARGETID...")
df_merged = pd.merge(df_hdu1, df_hdu2, on='TARGETID', how='inner')
print(f"Common targets in both HDUs: {len(df_merged)}")

# Apply quality cuts
print("\nApplying quality cuts...")

mask = (
    (df_merged['SPECTYPE'] == 'GALAXY') & 
    (df_merged['ZWARN'] == 0) &
    (df_merged['Z'] > 0.02) &
    (df_merged['Z'] < 0.31) &
    (df_merged['DN4000'] >= 0) &
    (df_merged['DN4000'] <= 5) &
    df_merged.notna().all(axis=1)
)

df = df_merged[mask].reset_index(drop=True)
df = df.sample(frac=1, random_state=32).reset_index(drop=True)

print(f"After all cuts: {df.shape[0]} galaxies")
print(f"Cut statistics:")
print(f"  Z in [0.02, 0.31]: {((df_merged['Z'] > 0.02) & (df_merged['Z'] < 0.31)).sum()}")
print(f"  DN4000 in [0, 5]: {((df_merged['DN4000'] >= 0) & (df_merged['DN4000'] <= 5)).sum()}")
print(f"  All cuts combined: {mask.sum()}")

In [ ]:
# Build properties (9 dimensions)
redshift = df['Z'].values
logM = df['LOGMSTAR'].values
logSFR = np.log10(np.abs(df['SFR'].values))
dn4000 = df['DN4000'].values
Av = df['AV'].values
hbeta_flux = np.log10(1 + df['HBETA_FLUX'].values)
oiii_flux = np.log10(1 + df['OIII_5007_FLUX'].values)
halpha_flux = np.log10(1 + df['HALPHA_FLUX'].values)
nii_flux = np.log10(1 + df['NII_6584_FLUX'].values)

unnorm_props = np.column_stack([redshift, logM, logSFR, dn4000, Av, hbeta_flux, oiii_flux, halpha_flux, nii_flux])

# Check for non-finite values
if not np.isfinite(unnorm_props).all():
    n_bad = (~np.isfinite(unnorm_props)).sum()
    print(f"Warning: {n_bad} non-finite values in properties, filtering...")
    finite_mask2 = np.isfinite(unnorm_props).all(axis=1)
    df = df[finite_mask2].reset_index(drop=True)
    unnorm_props = unnorm_props[finite_mask2]
    redshift, logM, logSFR, dn4000, Av, hbeta_flux, oiii_flux, halpha_flux, nii_flux = unnorm_props.T

print(f"After property filtering: {df.shape[0]} galaxies")

In [ ]:
# dr2_rgb function from John Wu
def sdss_rgb(imgs, bands, scales=None, m=0.02):
    rgbscales = {'u': (2,1.5),
                 'g': (2,2.5),
                 'r': (1,1.5),
                 'i': (0,1.0),
                 'z': (0,0.4),
                 }
    if scales is not None:
        rgbscales.update(scales)

    I = 0
    for img,band in zip(imgs, bands):
        plane,scale = rgbscales[band]
        img = np.maximum(0, img * scale + m)
        I = I + img
    I /= len(bands)
        
    Q = 20
    fI = np.arcsinh(Q * I) / np.sqrt(Q)
    I += (I == 0.) * 1e-6
    H,W = I.shape
    rgb = np.zeros((H,W,3), np.float32)
    for img,band in zip(imgs, bands):
        plane,scale = rgbscales[band]
        rgb[:,:,plane] = (img * scale + m) * fI / I

    rgb = np.clip(rgb, 0, 1)
    return rgb

def dr2_rgb(rimgs, bands, **ignored):
    return sdss_rgb(rimgs, bands, scales=dict(g=(2,6.0), r=(1,3.4), z=(0,2.2)), m=0.03)

# Cross-reference with h5 file, extract images, build datasets

In [ ]:
# Open H5 file and cross-reference with FITS
h5_file = '../astroclip_desi.1.1.5.h5'
f = h5py.File(h5_file, 'r')

all_h5_targetids = []
for group_key in f.keys():
    all_h5_targetids.append(f[group_key]['targetids'][:])
all_h5_targetids = np.concatenate(all_h5_targetids)

print(f"Total galaxies in h5 file: {len(all_h5_targetids)}")
print(f"Unique galaxies in h5 file: {len(np.unique(all_h5_targetids))}")

# Cross-reference
fits_targetids = df['TARGETID'].values
h5_targetids_set = set(all_h5_targetids)
in_h5 = np.array([tid in h5_targetids_set for tid in fits_targetids])
df_final = df[in_h5].reset_index(drop=True)

print(f"\nGalaxies in both FITS and h5: {len(df_final)}")

# Build property arrays from df_final (9 dimensions) with log(1+flux) for emission lines
redshift_final = df_final['Z'].values
logM_final = df_final['LOGMSTAR'].values
logSFR_final = np.log10(np.abs(df_final['SFR'].values))
dn4000_final = df_final['DN4000'].values
Av_final = df_final['AV'].values
hbeta_flux_final = np.log10(1 + df_final['HBETA_FLUX'].values)
oiii_flux_final = np.log10(1 + df_final['OIII_5007_FLUX'].values)
halpha_flux_final = np.log10(1 + df_final['HALPHA_FLUX'].values)
nii_flux_final = np.log10(1 + df_final['NII_6584_FLUX'].values)

properties = np.column_stack([
    redshift_final, logM_final, logSFR_final, dn4000_final, Av_final,
    hbeta_flux_final, oiii_flux_final, halpha_flux_final, nii_flux_final
])

# Normalize properties
prop_mean = properties.mean(axis=0)
prop_std = properties.std(axis=0)
tab_dataset = (properties - prop_mean) / prop_std

print(f"\nProperties shape: {tab_dataset.shape}")
print(f"Property means: {prop_mean}")
print(f"Property stds: {prop_std}")

# Prepare photometry
print("\nPreparing photometry conditioning (5 bands: G, R, Z, W1, W2)...")
photometry_array = np.column_stack([
    df_final['FLUX_G'].values,
    df_final['FLUX_R'].values,
    df_final['FLUX_Z'].values,
    df_final['FLUX_W1'].values,
    df_final['FLUX_W2'].values
])

mags = 22.5 - 2.5 * np.log10(np.maximum(photometry_array, 1e-10))
mags_mean = mags.mean(axis=0)
mags_std = mags.std(axis=0)
phot_dataset = (mags - mags_mean) / mags_std

print(f"Photometry shape: {phot_dataset.shape}")

In [ ]:
# Extract images from h5 file for df_final target IDs
final_targetids = df_final['TARGETID'].values
n_final = len(final_targetids)

# Create a mapping from targetid to h5 group and index
targetid_to_location = {}
for group_key in f.keys():
    group_targetids = f[group_key]['targetids'][:]
    for idx, tid in enumerate(group_targetids):
        targetid_to_location[tid] = (group_key, idx)

# Pre-allocate arrays
image_size = 152
images_array = np.zeros((n_final, image_size, image_size, 3), dtype=np.float32)

# Extract data
print("Extracting images from h5 file...")
for i, tid in enumerate(tqdm(final_targetids)):
    if tid in targetid_to_location:
        group_key, idx = targetid_to_location[tid]
        images_array[i] = f[group_key]['images'][idx]
    else:
        print(f"Warning: targetid {tid} not found in h5 file")

print(f"\nOriginal images shape: {images_array.shape}")

In [ ]:
# Pre-compute dr2_rgb transformations
print("Pre-computing dr2_rgb transformations...")
images_rgb = np.zeros((n_final, 3, image_size, image_size), dtype=np.float32)
for i in tqdm(range(n_final), desc="Applying dr2_rgb"):
    img_grz = images_array[i]  # (H, W, 3) in g,r,z order
    img_rgb = dr2_rgb(img_grz.transpose(2, 0, 1), bands=["g", "r", "z"])  # (H, W, 3)
    images_rgb[i] = img_rgb.transpose(2, 0, 1)  # (3, H, W) for PyTorch

print(f"Transformed RGB images shape: {images_rgb.shape}")

f.close()

# SNR > 3 Selection and BPT Setup

In [ ]:
# Compute SNR for each BPT emission line: SNR = FLUX * sqrt(FLUX_IVAR)
snr_halpha = df_final['HALPHA_FLUX'].values * np.sqrt(np.maximum(df_final['HALPHA_FLUX_IVAR'].values, 0))
snr_hbeta = df_final['HBETA_FLUX'].values * np.sqrt(np.maximum(df_final['HBETA_FLUX_IVAR'].values, 0))
snr_oiii = df_final['OIII_5007_FLUX'].values * np.sqrt(np.maximum(df_final['OIII_5007_FLUX_IVAR'].values, 0))
snr_nii = df_final['NII_6584_FLUX'].values * np.sqrt(np.maximum(df_final['NII_6584_FLUX_IVAR'].values, 0))

# Require SNR > 3 in ALL four lines
snr_mask = (snr_halpha > 3) & (snr_hbeta > 3) & (snr_oiii > 3) & (snr_nii > 3)

print(f"SNR > 3 selection:")
print(f"  H-alpha:  {(snr_halpha > 3).sum():6d} / {len(snr_halpha)}")
print(f"  H-beta:   {(snr_hbeta > 3).sum():6d} / {len(snr_hbeta)}")
print(f"  [OIII]:   {(snr_oiii > 3).sum():6d} / {len(snr_oiii)}")
print(f"  [NII]:    {(snr_nii > 3).sum():6d} / {len(snr_nii)}")
print(f"  All four: {snr_mask.sum():6d} / {len(snr_mask)}")

# Store which indices (into df_final) pass the SNR cut
snr_indices = np.where(snr_mask)[0]
print(f"\n{len(snr_indices)} galaxies pass the SNR > 3 cut on all four BPT lines")

# Create Train/Val/Test Split and Load Models

In [ ]:
class GalaxyDataset(Dataset):
    def __init__(self, images_rgb, photometry, properties, indices):
        self.images_rgb = images_rgb
        self.photometry = photometry
        self.properties = properties
        self.indices = indices

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, idx):
        actual_idx = self.indices[idx]
        img_tensor = torch.from_numpy(self.images_rgb[actual_idx]).float()
        phot = torch.from_numpy(self.photometry[actual_idx]).float()
        prop = torch.from_numpy(self.properties[actual_idx]).float()
        return img_tensor, phot, prop

# Create train/val/test splits (same random_state as training)
num_samples = len(images_rgb)
indices = np.arange(num_samples)

train_idx, temp_idx = train_test_split(indices, test_size=0.2, random_state=32)
val_idx, test_idx = train_test_split(temp_idx, test_size=0.5, random_state=32)

print(f"Dataset split:")
print(f"  Training:   {len(train_idx):6d} samples")
print(f"  Validation: {len(val_idx):6d} samples")
print(f"  Test:       {len(test_idx):6d} samples")

test_dataset = GalaxyDataset(images_rgb, phot_dataset, tab_dataset, test_idx)

# Identify which test galaxies pass the SNR cut
# test_idx maps test position -> df_final index
# snr_mask is over df_final indices
snr_test_mask = snr_mask[test_idx]  # boolean mask over test galaxies
snr_test_positions = np.where(snr_test_mask)[0]  # positions within test_dataset

print(f"\nTest galaxies passing SNR > 3 cut: {len(snr_test_positions)} / {len(test_idx)}")

In [ ]:
# Model dimensions
num_vars = tab_dataset.shape[1]  # 9 properties
phot_dim = phot_dataset.shape[1]  # 5 photometry features
feature_dim = 256  # image encoder output dimension

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Load experimental image model (ImprovedConditionalFlowModel)
print("\nLoading experimental image model...")
image_model = ImprovedConditionalFlowModel(
    property_dim=num_vars,
    feature_dim=feature_dim,
    phot_dim=phot_dim,
    hidden_dim=512
).to(device)

image_model_path = "models/best_image_model_FSFV2_9props_sigma1eneg4.pth"
image_model.load_state_dict(torch.load(image_model_path, map_location=device))
image_model.eval()
print(f"Loaded image model from {image_model_path}")

# Load experimental photometry model (ImprovedPhotometryConditionalFlowModelSimple)
print("\nLoading experimental photometry model...")
phot_model = ImprovedPhotometryConditionalFlowModelSimple(
    phot_dim=phot_dim,
    property_dim=num_vars,
    hidden_dim=512
).to(device)

phot_model_path = "models/best_photometry_model_FSFV2_9props_sigma1eneg4.pth"
phot_model.load_state_dict(torch.load(phot_model_path, map_location=device))
phot_model.eval()
print(f"Loaded photometry model from {phot_model_path}")

print("\n" + "="*60)
print("Both experimental models loaded successfully!")
print("="*60)

# Ground-Truth BPT Diagram (Kauffmann et al. 2003)

In [ ]:
# Compute ground-truth BPT coordinates for SNR-selected test galaxies
# Use raw fluxes from df_final for test galaxies that pass SNR cut
bpt_df_indices = test_idx[snr_test_positions]  # df_final indices

halpha_true = df_final.iloc[bpt_df_indices]['HALPHA_FLUX'].values
hbeta_true = df_final.iloc[bpt_df_indices]['HBETA_FLUX'].values
oiii_true = df_final.iloc[bpt_df_indices]['OIII_5007_FLUX'].values
nii_true = df_final.iloc[bpt_df_indices]['NII_6584_FLUX'].values

log_nii_halpha_true = np.log10(nii_true / halpha_true)
log_oiii_hbeta_true = np.log10(oiii_true / hbeta_true)

# Kauffmann+03 demarcation curve
x_kauff = np.linspace(-1.5, 0.05, 200)
y_kauff = 0.61 / (x_kauff - 0.05) + 1.3

print(f"BPT coordinates computed for {len(bpt_df_indices)} SNR-selected test galaxies")
print(f"log(NII/Halpha) range: [{log_nii_halpha_true.min():.2f}, {log_nii_halpha_true.max():.2f}]")
print(f"log(OIII/Hbeta) range: [{log_oiii_hbeta_true.min():.2f}, {log_oiii_hbeta_true.max():.2f}]")

In [ ]:
# Plot ground-truth BPT diagram
fig, ax = plt.subplots(figsize=(8, 8))

ax.scatter(log_nii_halpha_true, log_oiii_hbeta_true, s=1, alpha=0.3, color='gray',
           label='Spectroscopic (ground truth)', rasterized=True)
ax.plot(x_kauff, y_kauff, 'k--', lw=2, label='Kauffmann+03')

ax.set_xlabel(r'log$_{10}$([NII] $\lambda$6584 / H$\alpha$)', fontsize=14)
ax.set_ylabel(r'log$_{10}$([OIII] $\lambda$5007 / H$\beta$)', fontsize=14)
ax.set_title(f'Ground-Truth BPT Diagram (SNR > 3, N = {len(bpt_df_indices)})', fontsize=15)
ax.set_xlim(-1.5, 0.5)
ax.set_ylim(-1.2, 1.5)
ax.legend(fontsize=11, loc='upper left', markerscale=10)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Model Predictions: Posterior Medians

In [ ]:
# Sample posteriors for SNR-selected test galaxies only
N_bpt = len(snr_test_positions)
print(f"Sampling posteriors for {N_bpt} SNR-selected test galaxies...")

# Property indices: 5=hbeta, 6=oiii, 7=halpha, 8=nii (in log(1+flux) space)
image_medians_bpt = np.zeros((N_bpt, num_vars))
phot_medians_bpt = np.zeros((N_bpt, num_vars))

for count, i in enumerate(tqdm(snr_test_positions, desc="Posterior medians")):
    img_tensor, phot_tensor, prop_norm = test_dataset[i]
    
    img_b = img_tensor.unsqueeze(0).to(device)
    phot_b = phot_tensor.unsqueeze(0).to(device)
    
    # Sample from image model
    samples_image = sample_properties_distribution_rk4(
        image_model, img=img_b, phot=phot_b,
        property_dim=num_vars, num_samples=1000, num_steps=100
    ) * prop_std + prop_mean
    
    # Sample from photometry model
    samples_phot = sample_properties_distribution_rk4(
        phot_model, phot=phot_b,
        property_dim=num_vars, num_samples=1000, num_steps=100
    ) * prop_std + prop_mean
    
    image_medians_bpt[count] = np.median(samples_image, axis=0)
    phot_medians_bpt[count] = np.median(samples_phot, axis=0)

print(f"\nDone! Posterior medians for {N_bpt} galaxies.")

In [ ]:
# Convert model predictions from log10(1+flux) back to raw flux, then compute BPT ratios
def log1p_to_bpt(predictions):
    """Convert log10(1+flux) predictions (columns 5-8) to BPT coordinates."""
    hbeta = 10**predictions[:, 5] - 1   # index 5
    oiii  = 10**predictions[:, 6] - 1   # index 6
    halpha = 10**predictions[:, 7] - 1  # index 7
    nii   = 10**predictions[:, 8] - 1   # index 8
    
    # Avoid log of zero/negative: clamp to small positive value
    hbeta = np.maximum(hbeta, 1e-10)
    oiii = np.maximum(oiii, 1e-10)
    halpha = np.maximum(halpha, 1e-10)
    nii = np.maximum(nii, 1e-10)
    
    log_nii_ha = np.log10(nii / halpha)
    log_oiii_hb = np.log10(oiii / hbeta)
    return log_nii_ha, log_oiii_hb

# BPT from posterior medians
log_nii_ha_img_med, log_oiii_hb_img_med = log1p_to_bpt(image_medians_bpt)
log_nii_ha_phot_med, log_oiii_hb_phot_med = log1p_to_bpt(phot_medians_bpt)

print("BPT coordinates from posterior medians computed.")

In [ ]:
# BPT diagrams: posterior medians (Image+Phot and Photometry-only)
fig, axes = plt.subplots(1, 2, figsize=(16, 8))

for ax, log_x, log_y, model_name, color in [
    (axes[0], log_nii_ha_img_med, log_oiii_hb_img_med, 'Image+Phot', 'blue'),
    (axes[1], log_nii_ha_phot_med, log_oiii_hb_phot_med, 'Photometry-Only', 'green')
]:
    # Ground truth in background
    ax.scatter(log_nii_halpha_true, log_oiii_hbeta_true, s=1, alpha=0.15, color='gray',
               label='Ground truth', rasterized=True)
    # Model predictions
    ax.scatter(log_x, log_y, s=1, alpha=0.3, color=color,
               label=f'{model_name} (posterior median)', rasterized=True)
    # Kauffmann+03 curve
    ax.plot(x_kauff, y_kauff, 'k--', lw=2, label='Kauffmann+03')
    
    ax.set_xlabel(r'log$_{10}$([NII] $\lambda$6584 / H$\alpha$)', fontsize=13)
    ax.set_ylabel(r'log$_{10}$([OIII] $\lambda$5007 / H$\beta$)', fontsize=13)
    ax.set_title(f'{model_name} - Posterior Medians', fontsize=14)
    ax.set_xlim(-1.5, 0.5)
    ax.set_ylim(-1.2, 1.5)
    ax.legend(fontsize=10, loc='upper left', markerscale=10)
    ax.grid(True, alpha=0.3)

fig.suptitle(f'BPT Diagrams: Posterior Median Predictions (N = {N_bpt})', fontsize=16, y=1.02)
plt.tight_layout()
plt.show()

# Model Predictions: Single Trajectories

In [ ]:
# Sample single trajectories for SNR-selected test galaxies
print(f"Sampling single trajectories for {N_bpt} SNR-selected test galaxies...")

image_single_bpt = np.zeros((N_bpt, num_vars))
phot_single_bpt = np.zeros((N_bpt, num_vars))

for count, i in enumerate(tqdm(snr_test_positions, desc="Single trajectories")):
    img_tensor, phot_tensor, prop_norm = test_dataset[i]
    
    img_b = img_tensor.unsqueeze(0).to(device)
    phot_b = phot_tensor.unsqueeze(0).to(device)
    
    # Single trajectory from image model
    pred_image = sample_properties_rk4(
        image_model, img=img_b, phot=phot_b,
        property_dim=num_vars, num_steps=100
    )
    image_single_bpt[count] = pred_image.cpu().numpy().squeeze() * prop_std + prop_mean
    
    # Single trajectory from photometry model
    pred_phot = sample_properties_rk4(
        phot_model, phot=phot_b,
        property_dim=num_vars, num_steps=100
    )
    phot_single_bpt[count] = pred_phot.cpu().numpy().squeeze() * prop_std + prop_mean

print(f"\nDone! Single-trajectory predictions for {N_bpt} galaxies.")

In [ ]:
# BPT from single trajectories
log_nii_ha_img_st, log_oiii_hb_img_st = log1p_to_bpt(image_single_bpt)
log_nii_ha_phot_st, log_oiii_hb_phot_st = log1p_to_bpt(phot_single_bpt)

# BPT diagrams: single trajectories (Image+Phot and Photometry-only)
fig, axes = plt.subplots(1, 2, figsize=(16, 8))

for ax, log_x, log_y, model_name, color in [
    (axes[0], log_nii_ha_img_st, log_oiii_hb_img_st, 'Image+Phot', 'blue'),
    (axes[1], log_nii_ha_phot_st, log_oiii_hb_phot_st, 'Photometry-Only', 'green')
]:
    # Ground truth in background
    ax.scatter(log_nii_halpha_true, log_oiii_hbeta_true, s=1, alpha=0.15, color='gray',
               label='Ground truth', rasterized=True)
    # Model predictions
    ax.scatter(log_x, log_y, s=1, alpha=0.3, color=color,
               label=f'{model_name} (single trajectory)', rasterized=True)
    # Kauffmann+03 curve
    ax.plot(x_kauff, y_kauff, 'k--', lw=2, label='Kauffmann+03')
    
    ax.set_xlabel(r'log$_{10}$([NII] $\lambda$6584 / H$\alpha$)', fontsize=13)
    ax.set_ylabel(r'log$_{10}$([OIII] $\lambda$5007 / H$\beta$)', fontsize=13)
    ax.set_title(f'{model_name} - Single Trajectory', fontsize=14)
    ax.set_xlim(-1.5, 0.5)
    ax.set_ylim(-1.2, 1.5)
    ax.legend(fontsize=10, loc='upper left', markerscale=10)
    ax.grid(True, alpha=0.3)

fig.suptitle(f'BPT Diagrams: Single-Trajectory Predictions (N = {N_bpt})', fontsize=16, y=1.02)
plt.tight_layout()
plt.show()